# 02 — Building the feedback intelligence platform

> **Applied Labs** · **Domain**: PE · **Difficulty**: Advanced · **Reading time**: ~40 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/09_feedback_intelligence/02_build.ipynb)

**Prerequisites**:
- [01 — Architecture](./01_architecture.ipynb)
- OpenAI API key for LLM-based extraction and summarisation

**What you will learn**:
- How to generate realistic synthetic feedback datasets with planted anomalies
- Building a production-grade aspect extraction pipeline with LLM calls
- Time-series trend analysis with anomaly detection and visualisation
- Rule-based alert systems that surface critical issues automatically
- Executive summary generation that converts data into actionable briefs
- End-to-end pipeline orchestration: ingest → extract → trend → alert → summarise

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "sentence-transformers>=2.2.0" "pandas>=2.0.0" "matplotlib>=3.7.0" "numpy>=1.24.0"

import os, json, re, hashlib, textwrap
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Any
from collections import defaultdict, Counter
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

# --- OpenAI setup ---
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = None
if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    print("OpenAI client initialised ✓")
else:
    print("⚠ OPENAI_API_KEY not set — using mock LLM responses")

print("Setup complete ✓")

## Section 1 — Synthetic feedback dataset

We generate **200 realistic feedback items** spanning 30 days with three
planted patterns to evaluate later:

1. **Normal baseline** — steady product quality, UI, and support mentions
2. **Emerging issue** — login failures start appearing on day 15 and
   accelerate (simulating a deployment bug)
3. **Seasonal pattern** — billing complaints spike at month-end (days 27–30)

Each item gets a realistic timestamp, source type, and multi-aspect review text.

In [ ]:
@dataclass
class FeedbackItem:
    """Normalised feedback item."""
    id: str
    text: str
    source: str
    timestamp: str
    metadata: Dict = field(default_factory=dict)

# --- Templates for synthetic generation ---
TEMPLATES: Dict[str, List[str]] = {
    "product_quality": [
        "The product quality is {adj}. {detail}",
        "Overall {adj} experience with the product. {detail}",
        "Build quality feels {adj}. {detail}",
    ],
    "customer_support": [
        "Customer support was {adj}. {detail}",
        "Had to contact support — {adj} experience. {detail}",
        "Support team was {adj}. {detail}",
    ],
    "ui_ux": [
        "The interface is {adj}. {detail}",
        "UI/UX feels {adj}. {detail}",
        "Dashboard design is {adj}. {detail}",
    ],
    "login": [
        "Login keeps failing. {detail}",
        "Can't sign in — getting timeout errors. {detail}",
        "Authentication is broken since the last update. {detail}",
        "Login page crashes every time. {detail}",
        "Two-factor auth stopped working. {detail}",
    ],
    "billing": [
        "Billing is confusing. {detail}",
        "Got charged twice this month. {detail}",
        "Invoice doesn't match my plan. {detail}",
        "Refund has been pending for weeks. {detail}",
    ],
    "performance": [
        "App performance is {adj}. {detail}",
        "Loading times are {adj}. {detail}",
        "The app {detail}",
    ],
    "onboarding": [
        "Setup process was {adj}. {detail}",
        "Getting started was {adj}. {detail}",
    ],
    "pricing": [
        "Pricing feels {adj} for what you get. {detail}",
        "The cost is {adj} compared to competitors. {detail}",
    ],
}

POSITIVE_ADJS = ["great", "excellent", "amazing", "fantastic", "smooth", "intuitive"]
NEGATIVE_ADJS = ["terrible", "awful", "frustrating", "poor", "slow", "confusing"]
NEUTRAL_ADJS = ["okay", "average", "decent", "fine", "acceptable"]

POSITIVE_DETAILS = [
    "Would recommend to anyone.", "Best in class.", "Impressed with the quality.",
    "Exactly what I needed.", "Works like a charm.", "Very satisfied overall.",
]
NEGATIVE_DETAILS = [
    "Very disappointed.", "Need to fix this ASAP.", "Considering switching providers.",
    "This has been ongoing for weeks.", "Totally unacceptable.", "Lost hours to this.",
]
NEUTRAL_DETAILS = [
    "Nothing special but gets the job done.", "Room for improvement.",
    "It works, but barely.", "Could be better.", "Meets minimum expectations.",
]

SOURCES = ["review", "ticket", "survey", "social"]
SOURCE_WEIGHTS = [0.4, 0.25, 0.2, 0.15]

def generate_dataset(n_items: int = 200, n_days: int = 30, seed: int = 42) -> List[FeedbackItem]:
    """Generate synthetic feedback with planted anomalies."""
    rng = np.random.RandomState(seed)
    base_date = datetime(2025, 1, 1)
    items: List[FeedbackItem] = []

    for i in range(n_items):
        day = rng.randint(0, n_days)
        timestamp = (base_date + timedelta(days=int(day), hours=rng.randint(0, 23),
                                           minutes=rng.randint(0, 59))).isoformat()
        source = rng.choice(SOURCES, p=SOURCE_WEIGHTS)

        # --- Choose aspect(s) based on day ---
        aspects_to_include: List[str] = []

        # Baseline aspects (always present)
        baseline_aspects = ["product_quality", "customer_support", "ui_ux",
                            "performance", "onboarding", "pricing"]
        aspects_to_include.append(rng.choice(baseline_aspects))

        # 30% chance of multi-aspect review
        if rng.random() < 0.3:
            second = rng.choice(baseline_aspects)
            if second != aspects_to_include[0]:
                aspects_to_include.append(second)

        # Emerging issue: login bugs start day 15, increasing probability
        if day >= 15:
            login_prob = min(0.15 + (day - 15) * 0.04, 0.7)
            if rng.random() < login_prob:
                aspects_to_include = ["login"]  # login dominates
                if rng.random() < 0.3:
                    aspects_to_include.append(rng.choice(baseline_aspects))

        # Seasonal: billing spikes days 27-30
        if day >= 27:
            if rng.random() < 0.6:
                aspects_to_include.append("billing")

        # --- Generate text ---
        parts: List[str] = []
        for asp in aspects_to_include:
            templates = TEMPLATES.get(asp, TEMPLATES["product_quality"])
            template = rng.choice(templates)

            if asp == "login":
                adj = rng.choice(NEGATIVE_ADJS)
                detail = rng.choice(NEGATIVE_DETAILS)
            elif asp == "billing" and day >= 27:
                adj = rng.choice(NEGATIVE_ADJS)
                detail = rng.choice(NEGATIVE_DETAILS)
            else:
                sentiment_roll = rng.random()
                if sentiment_roll < 0.45:
                    adj = rng.choice(POSITIVE_ADJS)
                    detail = rng.choice(POSITIVE_DETAILS)
                elif sentiment_roll < 0.75:
                    adj = rng.choice(NEGATIVE_ADJS)
                    detail = rng.choice(NEGATIVE_DETAILS)
                else:
                    adj = rng.choice(NEUTRAL_ADJS)
                    detail = rng.choice(NEUTRAL_DETAILS)

            parts.append(template.format(adj=adj, detail=detail))

        text = " ".join(parts)
        items.append(FeedbackItem(
            id=f"fb-{i:04d}", text=text, source=source,
            timestamp=timestamp, metadata={"day": day}
        ))

    return items

# --- Generate ---
dataset = generate_dataset(200, 30)
print(f"Generated {len(dataset)} feedback items over 30 days\n")

# Summary stats
source_counts = Counter(item.source for item in dataset)
day_counts = Counter(item.metadata["day"] for item in dataset)
print("Source distribution:")
for src, cnt in source_counts.most_common():
    print(f"  {src}: {cnt}")

print(f"\nSample items:")
for item in dataset[:5]:
    print(f"  [{item.source:>7}] Day {item.metadata['day']:>2}: "{item.text[:80]}..."")
print(f"\n✓ Dataset ready: {len(dataset)} items, {len(source_counts)} sources, {len(day_counts)} active days")

## Section 2 — Aspect extraction pipeline

We build the full extraction pipeline with two modes:
1. **LLM-based** (when OpenAI API key is available) — uses structured prompts
2. **Mock extractor** (fallback) — keyword-based simulation for offline use

The pipeline processes all 200 items and stores results in a DataFrame for
downstream analysis.

In [ ]:
@dataclass
class AspectResult:
    """Single extracted aspect-opinion pair."""
    feedback_id: str
    aspect: str
    opinion: str
    sentiment: str
    intensity: float
    evidence: str
    timestamp: str = ""

EXTRACTION_PROMPT = """Extract all aspect-level opinions from this customer feedback.

ASPECTS (use exactly): product_quality, customer_support, pricing, ui_ux,
performance, login, onboarding, billing, reliability, general

Return a JSON array:
[{{"aspect": "...", "opinion": "...", "sentiment": "positive|negative|neutral",
   "intensity": 0.0-1.0, "evidence": "verbatim quote"}}]

FEEDBACK: "{text}"
"""

def llm_extract(text: str) -> List[Dict]:
    """Extract aspects using OpenAI API."""
    if not client:
        return mock_extract(text)
    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": EXTRACTION_PROMPT.format(text=text)}],
            temperature=0.1,
            max_tokens=500,
        )
        content = resp.choices[0].message.content.strip()
        # Parse JSON from response
        match = re.search(r"\[.*\]", content, re.DOTALL)
        if match:
            return json.loads(match.group())
        return [{"aspect": "general", "opinion": content[:100], "sentiment": "neutral",
                 "intensity": 0.3, "evidence": text[:80]}]
    except Exception as e:
        print(f"  ⚠ LLM error: {e}")
        return mock_extract(text)

def mock_extract(text: str) -> List[Dict]:
    """Keyword-based aspect extraction (fallback)."""
    results = []
    text_lower = text.lower()

    aspect_signals = {
        "login": ["login", "sign in", "timeout", "authentication", "two-factor"],
        "billing": ["billing", "charged", "invoice", "refund", "payment"],
        "ui_ux": ["interface", "ui", "ux", "dashboard", "design", "navigation"],
        "customer_support": ["support", "help desk", "agent", "response time"],
        "performance": ["performance", "loading", "slow", "fast", "crash", "lag"],
        "pricing": ["pricing", "cost", "expensive", "cheap", "price"],
        "product_quality": ["product quality", "build quality", "quality"],
        "onboarding": ["setup", "onboarding", "getting started"],
    }
    positive_kw = {"great", "excellent", "amazing", "fantastic", "smooth", "intuitive",
                   "impressed", "recommend", "best", "satisfied", "charm"}
    negative_kw = {"terrible", "awful", "frustrating", "poor", "slow", "confusing",
                   "broken", "disappointed", "unacceptable", "failing", "crashes"}

    words = set(text_lower.split())
    is_pos = bool(words & positive_kw)
    is_neg = bool(words & negative_kw)

    matched = False
    for aspect, keywords in aspect_signals.items():
        if any(kw in text_lower for kw in keywords):
            sent = "positive" if is_pos and not is_neg else ("negative" if is_neg else "neutral")
            intensity = 0.75 if (is_pos or is_neg) else 0.4
            results.append({
                "aspect": aspect, "opinion": f"Customer feedback on {aspect}",
                "sentiment": sent, "intensity": intensity, "evidence": text[:80]
            })
            matched = True

    if not matched:
        sent = "positive" if is_pos else ("negative" if is_neg else "neutral")
        results.append({"aspect": "general", "opinion": "General feedback",
                        "sentiment": sent, "intensity": 0.4, "evidence": text[:80]})
    return results

# --- Process all 200 items ---
print("Extracting aspects from 200 feedback items...\n")
all_aspects: List[AspectResult] = []
extraction_errors = 0

for i, item in enumerate(dataset):
    try:
        extractions = llm_extract(item.text) if client else mock_extract(item.text)
        for ext in extractions:
            all_aspects.append(AspectResult(
                feedback_id=item.id,
                aspect=ext.get("aspect", "general"),
                opinion=ext.get("opinion", ""),
                sentiment=ext.get("sentiment", "neutral"),
                intensity=ext.get("intensity", 0.5),
                evidence=ext.get("evidence", ""),
                timestamp=item.timestamp,
            ))
    except Exception as e:
        extraction_errors += 1
    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1}/200 items ({len(all_aspects)} aspects extracted)")

# Convert to DataFrame
df_aspects = pd.DataFrame([vars(a) for a in all_aspects])
print(f"\nExtraction complete:")
print(f"  Total aspects extracted: {len(df_aspects)}")
print(f"  Unique aspects: {df_aspects['aspect'].nunique()}")
print(f"  Errors: {extraction_errors}")
print(f"\nAspect distribution:")
print(df_aspects["aspect"].value_counts().to_string())
print(f"\nSentiment distribution:")
print(df_aspects["sentiment"].value_counts().to_string())
print(f"\n✓ Aspect extraction pipeline complete")

## Section 3 — Trend analysis engine

With aspect data extracted, we aggregate by **day × aspect** to build time
series. The trend engine computes:
- Daily mention counts per aspect
- 7-day moving averages for smoothing
- Z-score anomaly detection
- Trend slopes for each aspect

We expect to see: login bugs emerging from day 15, billing spike at month-end.

In [ ]:
class TrendEngine:
    """Analyse temporal trends in aspect data."""

    def __init__(self, window: int = 7, z_threshold: float = 2.0):
        self.window = window
        self.z_threshold = z_threshold

    def build_timeseries(self, df: pd.DataFrame, n_days: int = 30) -> pd.DataFrame:
        """Aggregate aspects into daily counts."""
        df = df.copy()
        df["date"] = pd.to_datetime(df["timestamp"]).dt.date
        date_range = pd.date_range("2025-01-01", periods=n_days, freq="D").date

        pivot = df.groupby(["date", "aspect"]).size().unstack(fill_value=0)
        # Reindex to ensure all days present
        pivot = pivot.reindex(date_range, fill_value=0)
        pivot.index = pd.to_datetime(pivot.index)
        return pivot

    def detect_anomalies(self, ts: pd.DataFrame) -> Dict[str, pd.DataFrame]:
        """Compute moving averages and z-score anomalies per aspect."""
        results = {}
        for aspect in ts.columns:
            series = ts[aspect].astype(float)
            ma = series.rolling(window=self.window, min_periods=3).mean()
            std = series.rolling(window=self.window, min_periods=3).std().replace(0, 1)
            z = (series - ma) / std
            results[aspect] = pd.DataFrame({
                "count": series, "moving_avg": ma, "z_score": z,
                "is_anomaly": z > self.z_threshold
            }, index=ts.index)
        return results

    def compute_slopes(self, ts: pd.DataFrame, recent_days: int = 7) -> Dict[str, float]:
        """Compute trend slope (recent avg - earlier avg) / days."""
        slopes = {}
        for aspect in ts.columns:
            vals = ts[aspect].values.astype(float)
            if len(vals) >= recent_days * 2:
                recent = vals[-recent_days:].mean()
                earlier = vals[:recent_days].mean()
                slopes[aspect] = round((recent - earlier) / recent_days, 3)
            else:
                slopes[aspect] = 0.0
        return slopes

# --- Run trend analysis ---
engine = TrendEngine(window=7, z_threshold=2.0)
ts = engine.build_timeseries(df_aspects, n_days=30)
anomalies = engine.detect_anomalies(ts)
slopes = engine.compute_slopes(ts)

print("Trend slopes (mentions/day change, recent vs early):\n")
for asp, slope in sorted(slopes.items(), key=lambda x: x[1], reverse=True):
    arrow = "↑↑" if slope > 0.5 else ("↑" if slope > 0.1 else ("→" if slope > -0.1 else "↓"))
    print(f"  {arrow} {asp:<25} {slope:+.3f}")

# --- Visualise top aspects ---
top_aspects = sorted(slopes.keys(), key=lambda a: ts[a].sum() if a in ts.columns else 0, reverse=True)[:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
for ax, aspect in zip(axes.flat, top_aspects):
    if aspect in anomalies:
        data = anomalies[aspect]
        ax.plot(data.index, data["count"], alpha=0.6, label="daily count")
        ax.plot(data.index, data["moving_avg"], color="black", linestyle="--",
                linewidth=1, label="7-day MA")
        anom_mask = data["is_anomaly"]
        if anom_mask.any():
            ax.scatter(data.index[anom_mask], data["count"][anom_mask],
                       color="red", s=60, zorder=5, marker="^", label="anomaly")
        ax.set_title(f"{aspect} (slope: {slopes.get(aspect, 0):+.3f})")
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

plt.suptitle("Aspect Trends — 30-Day Analysis", fontsize=14)
plt.tight_layout()
plt.show()

# Anomaly summary
print("\nAnomaly summary:")
total_anomalies = 0
for aspect, data in anomalies.items():
    n_anom = data["is_anomaly"].sum()
    if n_anom > 0:
        first = data.index[data["is_anomaly"]].min().strftime("%Y-%m-%d")
        print(f"  {aspect}: {n_anom} anomalous days, first on {first}")
        total_anomalies += n_anom
print(f"\n✓ Trend analysis complete — {total_anomalies} anomalies detected across {len(anomalies)} aspects")

## Section 4 — Alert system

The alert engine converts trend anomalies into structured, actionable alerts.
Three alert types:
- **Volume spike** — sudden increase in mentions (z-score > threshold)
- **Sentiment drop** — aspect sentiment trending negative
- **Emerging aspect** — new aspect appears that wasn't seen before

Alerts are prioritised by severity and recency.

In [ ]:
@dataclass
class Alert:
    """Structured alert from the feedback intelligence system."""
    alert_type: str
    aspect: str
    severity: str
    message: str
    timestamp: str
    score: float = 0.0

class AlertEngine:
    """Generate alerts from trend and sentiment data."""

    def __init__(self, z_critical: float = 3.0, z_warning: float = 2.0,
                 sentiment_threshold: float = -0.3):
        self.z_critical = z_critical
        self.z_warning = z_warning
        self.sentiment_threshold = sentiment_threshold

    def generate(self, anomalies: Dict[str, pd.DataFrame],
                 df_aspects: pd.DataFrame, slopes: Dict[str, float]) -> List[Alert]:
        """Generate all alerts."""
        alerts: List[Alert] = []

        # --- Volume spike alerts ---
        for aspect, data in anomalies.items():
            anom_days = data[data["is_anomaly"]]
            for date, row in anom_days.iterrows():
                severity = "critical" if row["z_score"] > self.z_critical else "warning"
                alerts.append(Alert(
                    alert_type="volume_spike", aspect=aspect, severity=severity,
                    message=f"{aspect}: {int(row['count'])} mentions on {date.strftime('%Y-%m-%d')} "
                            f"(z={row['z_score']:.1f}, MA={row['moving_avg']:.1f})",
                    timestamp=date.strftime("%Y-%m-%d"),
                    score=row["z_score"] * (2 if severity == "critical" else 1),
                ))

        # --- Sentiment drop alerts ---
        aspect_sentiment = df_aspects.groupby("aspect").apply(
            lambda g: (g["sentiment"] == "negative").mean() - (g["sentiment"] == "positive").mean()
        )
        for aspect, net_sentiment in aspect_sentiment.items():
            if net_sentiment > 0.3:  # more negative than positive
                alerts.append(Alert(
                    alert_type="sentiment_drop", aspect=aspect, severity="warning",
                    message=f"{aspect}: net negative sentiment ({net_sentiment:.0%} more negative than positive)",
                    timestamp=datetime.now().strftime("%Y-%m-%d"),
                    score=net_sentiment * 50,
                ))

        # --- Emerging aspect alerts (strong positive slope) ---
        for aspect, slope in slopes.items():
            if slope > 0.3:
                alerts.append(Alert(
                    alert_type="emerging_aspect", aspect=aspect, severity="info",
                    message=f"{aspect}: emerging trend (slope={slope:+.3f}/day)",
                    timestamp=datetime.now().strftime("%Y-%m-%d"),
                    score=slope * 30,
                ))

        # Sort by score
        alerts.sort(key=lambda a: a.score, reverse=True)
        return alerts

# --- Generate alerts ---
alert_engine = AlertEngine()
alerts = alert_engine.generate(anomalies, df_aspects, slopes)

# Display
icons = {"critical": "🔴", "warning": "🟡", "info": "🔵"}
print(f"Generated {len(alerts)} alerts\n")
print("Top 15 alerts by priority:\n")
for a in alerts[:15]:
    icon = icons.get(a.severity, "⚪")
    print(f"  {icon} [{a.severity:>8}] [P{a.score:>5.1f}] {a.message}")

# Summary
alert_counts = Counter(a.severity for a in alerts)
print(f"\nAlert summary: {alert_counts.get('critical', 0)} critical, "
      f"{alert_counts.get('warning', 0)} warning, {alert_counts.get('info', 0)} info")
print(f"✓ Alert engine operational")

## Section 5 — Executive summary generator

The summariser aggregates all pipeline outputs into a concise executive brief.
With an OpenAI API key, it uses GPT-4o-mini for natural language generation.
Without one, it falls back to a template-based approach.

The summary covers: top issues, emerging trends, and recommended actions.

In [ ]:
SUMMARY_SYSTEM_PROMPT = """You are a customer feedback analyst. Generate a concise executive
summary (5-7 sentences) from the data provided. Include:
1. Top 3 issues by volume and severity with specific numbers
2. Emerging trends (what's getting worse or better)
3. 2-3 specific recommended actions, prioritised by impact

Use professional tone. Be specific with numbers. Be actionable."""

def build_summary_context(df_aspects: pd.DataFrame, alerts: List[Alert],
                          slopes: Dict[str, float]) -> str:
    """Build context string for the summariser."""
    lines = []

    # Aspect counts
    counts = df_aspects["aspect"].value_counts()
    lines.append("ASPECT VOLUMES (total mentions):")
    for asp, cnt in counts.head(10).items():
        sent_dist = df_aspects[df_aspects["aspect"] == asp]["sentiment"].value_counts().to_dict()
        lines.append(f"  {asp}: {cnt} mentions — {sent_dist}")

    # Trends
    lines.append("\nTREND SLOPES (change in mentions/day):")
    for asp, slope in sorted(slopes.items(), key=lambda x: abs(x[1]), reverse=True)[:8]:
        direction = "increasing" if slope > 0.1 else ("decreasing" if slope < -0.1 else "stable")
        lines.append(f"  {asp}: {slope:+.3f}/day ({direction})")

    # Top alerts
    lines.append(f"\nTOP ALERTS ({len(alerts)} total):")
    for a in alerts[:10]:
        lines.append(f"  [{a.severity}] {a.message}")

    return "\n".join(lines)

def generate_executive_summary(df_aspects: pd.DataFrame, alerts: List[Alert],
                                slopes: Dict[str, float]) -> str:
    """Generate executive summary using LLM or template fallback."""
    context = build_summary_context(df_aspects, alerts, slopes)

    if client:
        try:
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": SUMMARY_SYSTEM_PROMPT},
                    {"role": "user", "content": f"Period: 2025-01-01 to 2025-01-30\n\n{context}"},
                ],
                temperature=0.3,
                max_tokens=500,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            print(f"  ⚠ LLM summary failed: {e}, using template fallback")

    # --- Template fallback ---
    counts = df_aspects["aspect"].value_counts()
    top_3 = list(counts.head(3).items())
    worsening = [(a, s) for a, s in slopes.items() if s > 0.2]
    worsening.sort(key=lambda x: x[1], reverse=True)
    critical = [a for a in alerts if a.severity == "critical"]

    parts = [
        f"📊 Executive Summary — January 2025 (30 days, {len(df_aspects)} aspect mentions)",
        f"{'=' * 65}",
        "",
        f"Top issues by volume:",
    ]
    for i, (asp, cnt) in enumerate(top_3, 1):
        neg_pct = (df_aspects[df_aspects["aspect"] == asp]["sentiment"] == "negative").mean()
        parts.append(f"  {i}. {asp}: {cnt} mentions ({neg_pct:.0%} negative)")

    if worsening:
        parts.append(f"\nEmerging trends (worsening):")
        for asp, slope in worsening[:3]:
            parts.append(f"  ⚠ {asp}: +{slope:.2f} mentions/day trend")

    parts.append(f"\n{len(critical)} critical alerts requiring immediate attention.")
    parts.append(f"\nRecommended actions:")
    if worsening:
        parts.append(f"  1. URGENT: Investigate {worsening[0][0]} — fastest growing issue")
    parts.append(f"  2. Review all {len(critical)} critical volume spike alerts")
    parts.append(f"  3. Schedule deep-dive on top 3 aspects with product team")

    return "\n".join(parts)

# --- Generate summary ---
summary = generate_executive_summary(df_aspects, alerts, slopes)
print(summary)
print("\n✓ Executive summary generated")

## Section 6 — End-to-end demo

Full pipeline on the 200-item dataset: **ingest → extract → trend → alert →
summarise**. We print the complete results: aspect frequency table, trend
charts, alert feed, and executive summary.

In [ ]:
print("=" * 70)
print("  FEEDBACK INTELLIGENCE PLATFORM — END-TO-END DEMO")
print("=" * 70)

# Step 1: Dataset summary
print(f"\n📥 STEP 1: Ingestion")
print(f"  {len(dataset)} feedback items from {len(set(i.source for i in dataset))} sources")
print(f"  Date range: {dataset[0].timestamp[:10]} to {dataset[-1].timestamp[:10]}")

# Step 2: Extraction summary
print(f"\n🔍 STEP 2: Aspect Extraction")
print(f"  {len(df_aspects)} aspect-opinion pairs extracted")
print(f"  Unique aspects: {df_aspects['aspect'].nunique()}")
print(f"\n  Aspect frequency table:")
freq = df_aspects.groupby("aspect").agg(
    count=("sentiment", "size"),
    positive=("sentiment", lambda x: (x == "positive").sum()),
    negative=("sentiment", lambda x: (x == "negative").sum()),
    neutral=("sentiment", lambda x: (x == "neutral").sum()),
).sort_values("count", ascending=False)
print(freq.to_string())

# Step 3: Trends
print(f"\n📈 STEP 3: Trend Analysis")
for asp, slope in sorted(slopes.items(), key=lambda x: x[1], reverse=True):
    arrow = "↑↑" if slope > 0.5 else ("↑" if slope > 0.1 else ("→" if slope > -0.1 else "↓"))
    anom_count = anomalies[asp]["is_anomaly"].sum() if asp in anomalies else 0
    print(f"  {arrow} {asp:<25} slope: {slope:+.3f}  anomalies: {anom_count}")

# Step 4: Alerts
print(f"\n🚨 STEP 4: Alert Feed ({len(alerts)} alerts)")
for a in alerts[:10]:
    icon = icons.get(a.severity, "⚪")
    print(f"  {icon} {a.message[:75]}")
if len(alerts) > 10:
    print(f"  ... and {len(alerts) - 10} more")

# Step 5: Executive summary
print(f"\n📊 STEP 5: Executive Summary")
print("─" * 50)
print(summary)
print("─" * 50)

# Final trend chart
fig, ax = plt.subplots(figsize=(14, 5))
for aspect in ts.columns:
    if ts[aspect].sum() > 5:
        ax.plot(ts.index, ts[aspect].rolling(3).mean(), label=aspect, alpha=0.8)
ax.set_title("All Aspect Trends — 30 Day Overview (3-day smoothed)")
ax.set_ylabel("Mentions / day")
ax.legend(fontsize=8, ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✓ End-to-end pipeline complete")

In [ ]:
# --- Pipeline integration check ---
integration_checks = {
    "dataset_size": len(dataset) == 200,
    "extraction_complete": len(df_aspects) > 100,
    "trends_computed": len(slopes) > 0,
    "alerts_generated": len(alerts) > 0,
    "summary_generated": len(summary) > 50,
}
for check, passed in integration_checks.items():
    status = "✓" if passed else "✗"
    print(f"  {status} {check}")
all_passed = all(integration_checks.values())
print(f"\n✓ All checks passed" if all_passed else "⚠ Some checks failed")

## Exercises

1. **Expand the dataset** — Add 100 more items with a new planted pattern:
   a competitor-mention trend starting on day 10. Track how quickly the system
   detects it and generates an alert.

2. **Real API extraction** — If you have an OpenAI key, swap `mock_extract`
   for `llm_extract` and compare extraction quality. Measure: aspect counts,
   sentiment agreement, and any hallucinated aspects.

3. **Custom alert rules** — Add a "sentiment reversal" alert that triggers
   when an aspect's sentiment flips from net-positive to net-negative within
   a 5-day window. Test on the billing aspect around month-end.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Synthetic data with planted anomalies enables rigorous pipeline testing |
| 2 | LLM extraction with mock fallback allows offline development and testing |
| 3 | Time-series aggregation by day × aspect reveals patterns invisible in raw text |
| 4 | Z-score anomaly detection is simple, configurable, and effective for feedback trends |
| 5 | Rule-based alerts with priority scoring create actionable, ranked output |
| 6 | Executive summaries convert hundreds of data points into 5–7 sentence briefs |
| 7 | End-to-end pipeline testing validates that all stages compose correctly |

## What's Next

In **03 — Evaluate**, we rigorously assess each pipeline stage: aspect
extraction accuracy, sentiment precision, trend detection recall, summary
quality, and cost analysis.

---

## References

1. OpenAI, *GPT-4o-mini: Efficient Structured Extraction*, 2024 — <https://platform.openai.com/docs/models>
2. Hu, M. & Liu, B., *Mining and Summarizing Customer Reviews*, KDD 2004 — <https://dl.acm.org/doi/10.1145/1014052.1014073>
3. Chandola, V. et al., *Anomaly Detection: A Survey*, ACM Computing Surveys, 2009 — <https://dl.acm.org/doi/10.1145/1541880.1541882>
4. Qualtrics, *XM Platform Documentation* — <https://www.qualtrics.com/experience-management/>